# Lateral Movement Detection: Rule-Based vs Graph-Based

This notebook loads the synthetic authentication dataset from Kaggle
(`danielpeng1995/synthetic-enterprise-auth-logs`), runs both detection methods, and reports independently calculated precision, recall, FPR, and confusion matrices.

**1. Load and validate the dataset**  
180,000 authentication events, 498 active users, 150 hosts, and 16,391 labeled MITRE ATT&CK T1078/T1021 attack chains.

**2. Rule-based detector**  
Flags new user-to-host connections and previously unseen source subnets.

**3. Graph-based detector**  
Uses edge novelty, path rarity, and host degree deviation, combined with logistic regression.

**4. Independent evaluation**  
Both methods are evaluated on a held-out test period using the same alert budget for a fair side-by-side comparison.

## 1. Load the dataset from Kaggle

The dataset is public: https://www.kaggle.com/datasets/danielpeng1995/synthetic-enterprise-auth-logs

`kagglehub` downloads it without needing a manual file upload. If Kaggle prompts for
authentication (some environments require it even for public datasets), run
`kagglehub.login()` in a new cell and follow the prompt, then re-run this cell.

In [29]:
!pip install -q kagglehub networkx scikit-learn

import glob
import numpy as np
import pandas as pd
import networkx as nx
import kagglehub
from collections import defaultdict
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from pathlib import Path

dataset_path = Path(
    kagglehub.dataset_download(
        "danielpeng1995/synthetic-enterprise-auth-logs"
    )
)

df = pd.read_csv(
    dataset_path / "synthetic_auth_events_180000.csv"
)

print(f"Loaded {len(df):,} authentication events")

Using Colab cache for faster access to the 'synthetic-enterprise-auth-logs' dataset.
Loaded 180,000 authentication events


In [30]:
df['timestamp'] = pd.to_datetime(df['timestamp'], format='mixed')
df = df.sort_values('timestamp').reset_index(drop=True)
df['day'] = (df['timestamp'] - df['timestamp'].min()).dt.days

DC = 'DC-01'

print(f'Total events: {len(df):,}')
print(f'Malicious events: {int(df.is_malicious.sum())}')
print(f'Attack chains: {df.attack_chain_id.nunique()}')
print(f'Date range: {df.timestamp.min()} to {df.timestamp.max()}')
df.head()

Total events: 180,000
Malicious events: 64860
Attack chains: 16391
Date range: 2026-05-01 00:00:00 to 2026-05-30 23:59:59


,event_id,timestamp,user_id,user_role,src_host,src_subnet,dst_host,auth_protocol,logon_type,auth_result,is_domain_controller_target,is_malicious,attack_chain_id,mitre_technique,day
0,1,2026-05-01 00:00:00,U0001,admin,SRV-001,10.99.0.0/24,DC-01,NTLM,Interactive,Success,True,False,NaN,NaN,0
1,2,2026-05-01 00:00:07,U0001,admin,SRV-001,10.99.0.0/24,SRV-001,Kerberos,Service,Success,False,False,NaN,NaN,0
2,3,2026-05-01 00:00:08,U0032,admin,WKS-0017,10.10.11.0/24,WKS-0017,NTLM,Network,Success,False,False,NaN,NaN,0
3,4,2026-05-01 00:00:14,U0001,admin,SRV-001,10.99.0.0/24,SRV-002,NTLM,Interactive,Success,False,False,NaN,NaN,0
4,5,2026-05-01 00:00:21,U0001,admin,SRV-001,10.99.0.0/24,SRV-003,Kerberos,Service,Success,False,False,NaN,NaN,0


## 2. Temporal train / fit / test split

Detection systems learn from history, so we split by time rather than randomly — otherwise
we'd be using the future to predict the past.

- **Baseline (day 0–7):** builds each user's normal host/subnet profile. Not scored.
- **Fit (day 7–15):** scored with the baseline profile, used only to fit the graph-based
  logistic regression weights.
- **Test (day 15–30):** held out. This is the only period either method is *evaluated* on.

In [31]:
BASELINE_END = df['timestamp'].min() + pd.Timedelta(days=7)
FIT_END = df['timestamp'].min() + pd.Timedelta(days=15)

baseline_df = df[df.timestamp < BASELINE_END]
fit_df = df[(df.timestamp >= BASELINE_END) & (df.timestamp < FIT_END)]
test_df = df[df.timestamp >= FIT_END]

print(f'Baseline : {len(baseline_df):>7,} events (profile-building only, not scored)')
print(f'Fit      : {len(fit_df):>7,} events, {int(fit_df.is_malicious.sum())} malicious '
      '(used to fit the graph-based weights)')
print(f'Test     : {len(test_df):>7,} events, {int(test_df.is_malicious.sum())} malicious '
      '(held out — this is what gets reported)')

Baseline :  42,000 events (profile-building only, not scored)
Fit      :  48,000 events, 22560 malicious (used to fit the graph-based weights)
Test     :  90,000 events, 42300 malicious (held out — this is what gets reported)


## 3. Rule-based baseline

Two rules, exactly as described on the poster:
1. **Flag first-time login to a host** for this user
2. **Flag login from a new / unseen subnet** for this user

Both are evaluated *causally* — in strict time order, using only what the detector has
observed so far for that user — and history keeps updating through the fit and test periods.

In [32]:
def score_novelty(history_df, score_df, seen_host, seen_subnet):
    novel_host_flags, novel_subnet_flags = [], []
    for row in score_df.itertuples():
        nh = row.dst_host not in seen_host[row.user_id]
        ns = row.src_subnet not in seen_subnet[row.user_id]
        novel_host_flags.append(nh)
        novel_subnet_flags.append(ns)
        seen_host[row.user_id].add(row.dst_host)
        seen_subnet[row.user_id].add(row.src_subnet)
    return np.array(novel_host_flags), np.array(novel_subnet_flags)

seen_host_rb = defaultdict(set)
seen_subnet_rb = defaultdict(set)
for row in baseline_df.itertuples():
    seen_host_rb[row.user_id].add(row.dst_host)
    seen_subnet_rb[row.user_id].add(row.src_subnet)

nh_fit, ns_fit = score_novelty(baseline_df, fit_df, seen_host_rb, seen_subnet_rb)
nh_test, ns_test = score_novelty(fit_df, test_df, seen_host_rb, seen_subnet_rb)

rule_flag_test = nh_test | ns_test

In [33]:
y_test = test_df['is_malicious'].to_numpy()
tn, fp, fn, tp = confusion_matrix(y_test, rule_flag_test).ravel()
precision_rb = tp / (tp + fp)
recall_rb = tp / (tp + fn)
fpr_rb = fp / (fp + tn)

print('=== RULE-BASED: test-period confusion matrix ===')
print(f'TP={tp}  FP={fp}  FN={fn}  TN={tn}')
print(f'Precision={precision_rb:.3f}  Recall={recall_rb:.3f}  FPR={fpr_rb:.5f}')

=== RULE-BASED: test-period confusion matrix ===
TP=21996  FP=10017  FN=20304  TN=37683
Precision=0.687  Recall=0.520  FPR=0.21000


## 4. Graph-based method

Three features per event:

- **N (edge novelty)** — same causal signal as Rule 1: is this user→host pair unseen so far?
- **R (path rarity)** — build an undirected host–host graph (two hosts linked if any user
  touched both), run one BFS from the domain controller to get every host's distance to DC,
  then flag `R=1` when an event's destination is *closer to DC than this user has ever gotten
  before*.
- **D (degree deviation)** — for each host, compute daily distinct-user counts during the
  baseline (mean/std), then z-score each test-day's actual count against that baseline
  (positive spikes only).

A logistic regression combines the three into one score, fit on the **fit period** and
applied to the held-out **test period**.

In [34]:
def build_host_graph(history_df):
    G = nx.Graph()
    G.add_nodes_from(pd.unique(history_df[['src_host', 'dst_host']].values.ravel()))
    by_user = history_df.groupby('user_id')['dst_host'].apply(lambda s: sorted(set(s)))
    for hosts in by_user:
        for i in range(len(hosts)):
            for j in range(i + 1, len(hosts)):
                G.add_edge(hosts[i], hosts[j])
    return G

def dist_to_dc(G):
    if DC not in G:
        return defaultdict(lambda: 99)
    return defaultdict(lambda: 99, nx.single_source_shortest_path_length(G, DC))

def user_baseline_min_dist(history_df, dist_map):
    footprint = history_df.groupby('user_id')['dst_host'].apply(set)
    return {u: min([dist_map[h] for h in hosts], default=99) for u, hosts in footprint.items()}

def host_daily_baseline(history_df):
    daily = history_df.groupby(['dst_host', 'day'])['user_id'].nunique().reset_index(name='n')
    stats = daily.groupby('dst_host')['n'].agg(['mean', 'std']).fillna(0)
    return stats['mean'].to_dict(), stats['std'].to_dict()

def compute_R_D(history_df, score_df, dist_map, user_min_dist, host_mean, host_std):
    r_vals, d_vals = [], []
    daily_actual = score_df.groupby(['dst_host', 'day'])['user_id'].nunique().to_dict()
    for row in score_df.itertuples():
        d_this = dist_map[row.dst_host]
        baseline_min = user_min_dist.get(row.user_id, 99)
        r_vals.append(1.0 if d_this < baseline_min else 0.0)
        actual_n = daily_actual.get((row.dst_host, row.day), 1)
        mean_h = host_mean.get(row.dst_host, actual_n)
        std_h = host_std.get(row.dst_host, 0.0)
        z = (actual_n - mean_h) / (std_h + 0.5)
        d_vals.append(max(0.0, z))
    return np.array(r_vals), np.array(d_vals)

In [35]:
G_baseline = build_host_graph(baseline_df)
dist_baseline = dist_to_dc(G_baseline)
user_min_dist_baseline = user_baseline_min_dist(baseline_df, dist_baseline)
host_mean_b, host_std_b = host_daily_baseline(baseline_df)

R_fit, D_fit = compute_R_D(baseline_df, fit_df, dist_baseline, user_min_dist_baseline, host_mean_b, host_std_b)
N_fit = nh_fit.astype(float)

hist_for_test = pd.concat([baseline_df, fit_df])
G_test = build_host_graph(hist_for_test)
dist_test = dist_to_dc(G_test)
user_min_dist_test = user_baseline_min_dist(hist_for_test, dist_test)
host_mean_t, host_std_t = host_daily_baseline(hist_for_test)

R_test, D_test = compute_R_D(hist_for_test, test_df, dist_test, user_min_dist_test, host_mean_t, host_std_t)
N_test = nh_test.astype(float)

In [36]:
X_fit = np.column_stack([N_fit, R_fit, D_fit])
y_fit = fit_df['is_malicious'].to_numpy()
X_test = np.column_stack([N_test, R_test, D_test])

clf = LogisticRegression(class_weight='balanced')
clf.fit(X_fit, y_fit)
scores_test = clf.predict_proba(X_test)[:, 1]

print('=== GRAPH-BASED: fitted logistic regression ===')
print(f'Coefficients  N={clf.coef_[0][0]:.3f}  R={clf.coef_[0][1]:.3f}  D={clf.coef_[0][2]:.3f}  '
      f'intercept={clf.intercept_[0]:.3f}')

=== GRAPH-BASED: fitted logistic regression ===
Coefficients  N=1.572  R=1.572  D=0.091  intercept=-1.374


### Match alert volume to the rule-based method

To compare fairly, we don't let the graph-based method win just by raising fewer alerts.
We threshold it to raise *exactly as many alerts* as the rule-based method did, then compare
who used that same review budget better.

**Note on reproducibility:** with so many events sharing identical (N, R, D) feature values,
thousands of scores tie exactly at the alert threshold. The tie-break below is made explicit
(round scores, then break ties by row index) so the exact same alerts are selected on every
run — without it, tiny floating-point jitter in the logistic regression's internals could
silently flip which tied events land just above vs. below the cutoff.

In [37]:
n_rule_alerts = int(rule_flag_test.sum())
scores_rounded = np.round(scores_test, 6)
order = np.lexsort((np.arange(len(scores_rounded)), -scores_rounded))
graph_flag_test = np.zeros(len(scores_test), dtype=bool)
graph_flag_test[order[:n_rule_alerts]] = True

tn2, fp2, fn2, tp2 = confusion_matrix(y_test, graph_flag_test).ravel()
precision_gb = tp2 / (tp2 + fp2)
recall_gb = tp2 / (tp2 + fn2)
fpr_gb = fp2 / (fp2 + tn2)

print(f'Alert volume matched to rule-based: {n_rule_alerts} alerts')
print('=== GRAPH-BASED: test-period confusion matrix ===')
print(f'TP={tp2}  FP={fp2}  FN={fn2}  TN={tn2}')
print(f'Precision={precision_gb:.3f}  Recall={recall_gb:.3f}  FPR={fpr_gb:.5f}')

Alert volume matched to rule-based: 32013 alerts
=== GRAPH-BASED: test-period confusion matrix ===
TP=26336  FP=5677  FN=15964  TN=42023
Precision=0.823  Recall=0.623  FPR=0.11901


## 5. Side-by-side results

In [38]:
results = pd.DataFrame({
    'Method': ['Rule-Based', 'Graph-Based'],
    'Precision': [precision_rb, precision_gb],
    'Recall': [recall_rb, recall_gb],
    'FPR': [fpr_rb, fpr_gb],
})
results

,Method,Precision,Recall,FPR
0,Rule-Based,0.687096,0.5200,0.210000
1,Graph-Based,0.822666,0.6226,0.119015


## 6. Reading the result

Under the same alert budget, the graph-based method outperforms the rule-based baseline on all three reported metrics in the held-out test period:

- Precision increases from **0.687** to **0.823**
- Recall increases from **0.520** to **0.623**
- FPR decreases from **0.210** to **0.119**

The graph-based detector identifies more malicious events while sending fewer benign events to analysts, even though both methods generate exactly **32,013 alerts**.

These findings apply only to this synthetic, attack-enriched dataset and should be interpreted as a controlled feasibility result rather than evidence of production performance.